<a href="https://colab.research.google.com/github/Bachbean/Predicting-the-Unpredictable/blob/main/scripts/L63_surrogate_prediction_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
from scipy.spatial import cKDTree


# ============================================================
# 1. LOAD DATA
# ============================================================

file_name = "https://raw.githubusercontent.com/Bachbean/Predicting-the-Unpredictable/refs/heads/main/scripts/L63_x.txt"

data_all = np.loadtxt(file_name, dtype=np.float64)
data = data_all[:30000]
data = np.asarray(data).squeeze()

if data.ndim != 1:
    raise ValueError("Data must be one-dimensional.")

if np.any(~np.isfinite(data)):
    raise ValueError("Data contains NaN or infinite values.")

print("Number of data points:", len(data))


# ============================================================
# 2. SETTINGS
# ============================================================

# From your AMI/FNN analysis
tau = 16
m = 3

# Temporal exclusion window
theiler_window = 32


# ------------------------------------------------------------
# Lyapunov settings
# ------------------------------------------------------------

max_k = 100

fit_start = 5
fit_end = 55

dt = 1.0


# ------------------------------------------------------------
# Nonlinear prediction settings
# ------------------------------------------------------------

# Number of nearby reconstructed states used for prediction
k_neighbors = 10

# Predict several short-term horizons
prediction_horizons = [1, 5, 10, 20, 40]

# Using a subset speeds up the surrogate calculations
# 2000 is plenty with ~10,000 points
n_prediction_points = 2000

# Number of phase-randomized surrogate data sets
n_surrogates = 99

# Fixed seed makes results reproducible
rng = np.random.default_rng(12345)


# ============================================================
# 3. DELAY EMBEDDING
# ============================================================

def delay_embed(data, m, tau):

    N = len(data) - (m - 1) * tau

    if N <= 0:
        raise ValueError(
            "Embedding parameters are too large for the data."
        )

    X = np.empty((N, m))

    for j in range(m):

        X[:, j] = data[
            j * tau:
            j * tau + N
        ]

    return X


# ============================================================
# 4. LARGEST LYAPUNOV EXPONENT
#
# IMPORTANT:
# We calculate this for the REAL DATA as evidence of
# sensitive dependence.
#
# We do NOT compare surrogate lambda values anymore.
# ============================================================

def calculate_lyapunov(
    data,
    tau,
    m,
    theiler_window,
    max_k,
    fit_start,
    fit_end,
    dt=1.0
):

    X = delay_embed(
        data,
        m,
        tau
    )

    usable_points = len(X) - max_k

    if usable_points <= 10:
        raise ValueError(
            "Not enough data for Lyapunov analysis."
        )

    X_search = X[:usable_points]

    tree = cKDTree(X_search)

    number_candidates = min(
        max(
            50,
            2 * theiler_window + 20
        ),
        usable_points
    )

    distances, indices = tree.query(
        X_search,
        k=number_candidates
    )


    nearest_neighbor = np.full(
        usable_points,
        -1,
        dtype=int
    )


    for i in range(usable_points):

        for distance, candidate in zip(
            np.atleast_1d(distances[i]),
            np.atleast_1d(indices[i])
        ):

            if candidate >= usable_points:
                continue

            # Exclude nearby points in TIME
            if abs(candidate - i) <= theiler_window:
                continue

            if (
                not np.isfinite(distance)
                or distance <= 0
            ):
                continue

            nearest_neighbor[i] = candidate
            break


    reference_indices = np.where(
        nearest_neighbor >= 0
    )[0]

    neighbor_indices = nearest_neighbor[
        reference_indices
    ]


    if len(reference_indices) < 10:

        raise ValueError(
            "Too few valid nearest-neighbor pairs."
        )


    mean_log_divergence = np.full(
        max_k,
        np.nan
    )


    # --------------------------------------------------------
    # Follow nearby states forward through time
    # --------------------------------------------------------

    for k in range(max_k):

        A = X[
            reference_indices + k
        ]

        B = X[
            neighbor_indices + k
        ]

        separation = np.linalg.norm(
            A - B,
            axis=1
        )

        valid = (
            np.isfinite(separation)
            &
            (separation > 0)
        )

        if np.any(valid):

            mean_log_divergence[k] = np.mean(
                np.log(
                    separation[valid]
                )
            )


    # --------------------------------------------------------
    # Fit known linear region
    # --------------------------------------------------------

    k_values = np.arange(max_k)

    fit_mask = (
        (k_values >= fit_start)
        &
        (k_values < fit_end)
        &
        np.isfinite(
            mean_log_divergence
        )
    )

    x_fit = (
        k_values[fit_mask]
        * dt
    )

    y_fit = mean_log_divergence[
        fit_mask
    ]


    slope, intercept = np.polyfit(
        x_fit,
        y_fit,
        1
    )

    predicted = (
        slope * x_fit
        + intercept
    )


    ss_res = np.sum(
        (y_fit - predicted) ** 2
    )

    ss_tot = np.sum(
        (
            y_fit
            - np.mean(y_fit)
        ) ** 2
    )

    if ss_tot > 0:

        r_squared = (
            1
            -
            ss_res / ss_tot
        )

    else:

        r_squared = np.nan


    return slope, r_squared


# ============================================================
# 5. PHASE-RANDOMIZED SURROGATE
#
# Preserves:
#   - Fourier amplitudes
#   - power spectrum
#   - linear autocorrelation structure
#
# Randomizes:
#   - Fourier phase relationships
# ============================================================

def phase_randomized_surrogate(
    data,
    rng
):

    data = np.asarray(data)

    N = len(data)

    spectrum = np.fft.rfft(data)

    surrogate_spectrum = (
        spectrum.copy()
    )


    # --------------------------------------------------------
    # Frequencies whose phases may be randomized
    # --------------------------------------------------------

    if N % 2 == 0:

        # Do not modify:
        # index 0 = DC
        # final index = Nyquist frequency
        phase_indices = np.arange(
            1,
            len(spectrum) - 1
        )

    else:

        phase_indices = np.arange(
            1,
            len(spectrum)
        )


    random_phases = rng.uniform(
        0,
        2 * np.pi,
        len(phase_indices)
    )


    surrogate_spectrum[
        phase_indices
    ] = (
        np.abs(
            spectrum[
                phase_indices
            ]
        )
        *
        np.exp(
            1j
            * random_phases
        )
    )


    # Preserve DC exactly
    surrogate_spectrum[0] = (
        spectrum[0]
    )


    # Preserve Nyquist exactly
    if N % 2 == 0:

        surrogate_spectrum[-1] = (
            spectrum[-1]
        )


    surrogate = np.fft.irfft(
        surrogate_spectrum,
        n=N
    )

    return surrogate


# ============================================================
# 6. NONLINEAR LOCAL PREDICTION ERROR
#
# Basic idea:
#
# If two reconstructed states are very similar,
#
#       X_i ~ X_j
#
# then in a deterministic system their short-term futures
# should also be similar.
#
# We predict the future from the futures of nearby states.
# ============================================================

def nonlinear_prediction_error(
    data,
    tau,
    m,
    theiler_window,
    k_neighbors,
    horizons,
    n_prediction_points=2000,
    query_seed=2026
):

    data = np.asarray(data)

    max_horizon = max(
        horizons
    )


    # --------------------------------------------------------
    # Reconstruct state space
    # --------------------------------------------------------

    X = delay_embed(
        data,
        m,
        tau
    )


    # X[i] contains
    #
    # x[i],
    # x[i+tau],
    # ...
    #
    # Therefore the time corresponding to the LAST
    # coordinate of X[i] is:
    #
    # i + (m-1)*tau
    # --------------------------------------------------------

    state_times = (
        np.arange(len(X))
        +
        (m - 1) * tau
    )


    # Need enough future samples for the largest horizon

    usable_points = (
        len(X)
        -
        max_horizon
    )

    X_search = X[
        :usable_points
    ]

    state_times = state_times[
        :usable_points
    ]


    if usable_points <= 100:

        raise ValueError(
            "Too few usable reconstructed states."
        )


    # --------------------------------------------------------
    # Build nearest-neighbor tree
    # --------------------------------------------------------

    tree = cKDTree(
        X_search
    )


    # --------------------------------------------------------
    # Choose fixed query locations
    #
    # Same time indices are used for every surrogate,
    # making the comparison fair.
    # --------------------------------------------------------

    query_rng = np.random.default_rng(
        query_seed
    )


    if (
        n_prediction_points
        >= usable_points
    ):

        query_indices = np.arange(
            usable_points
        )

    else:

        query_indices = np.sort(
            query_rng.choice(
                usable_points,
                size=n_prediction_points,
                replace=False
            )
        )


    # Search more candidates than k because many will
    # be rejected by the Theiler window

    number_candidates = min(
        max(
            50,
            4 * k_neighbors,
            2 * theiler_window + 20
        ),
        usable_points
    )


    distances, indices = tree.query(
        X_search[
            query_indices
        ],
        k=number_candidates
    )


    squared_errors = {
        h: []
        for h in horizons
    }


    successful_queries = 0


    # ========================================================
    # FIND LOCAL NEIGHBORS AND PREDICT FUTURE
    # ========================================================

    for row, i in enumerate(
        query_indices
    ):

        neighbors = []


        for candidate in np.atleast_1d(
            indices[row]
        ):

            if candidate >= usable_points:
                continue


            # Exclude states close in TIME
            if (
                abs(
                    state_times[candidate]
                    -
                    state_times[i]
                )
                <= theiler_window
            ):

                continue


            neighbors.append(
                candidate
            )


            if (
                len(neighbors)
                >= k_neighbors
            ):

                break


        if (
            len(neighbors)
            < k_neighbors
        ):

            continue


        neighbors = np.array(
            neighbors,
            dtype=int
        )

        successful_queries += 1


        # ----------------------------------------------------
        # Predict each future horizon
        # ----------------------------------------------------

        for h in horizons:

            neighbor_future_values = data[
                state_times[
                    neighbors
                ]
                + h
            ]


            # Local constant prediction:
            # average future of nearby states

            prediction = np.mean(
                neighbor_future_values
            )


            actual = data[
                state_times[i]
                + h
            ]


            squared_errors[h].append(
                (
                    prediction
                    -
                    actual
                ) ** 2
            )


    # ========================================================
    # NORMALIZED RMSE
    # ========================================================

    data_scale = np.std(
        data
    )


    if data_scale == 0:

        raise ValueError(
            "Data have zero variance."
        )


    nrmse = {}


    for h in horizons:

        errors = np.asarray(
            squared_errors[h]
        )


        if len(errors) == 0:

            nrmse[h] = np.nan

        else:

            rmse = np.sqrt(
                np.mean(errors)
            )

            nrmse[h] = (
                rmse
                /
                data_scale
            )


    # --------------------------------------------------------
    # Single summary statistic:
    # average NRMSE across horizons
    #
    # LOWER = more locally predictable
    # --------------------------------------------------------

    values = np.array(
        list(
            nrmse.values()
        )
    )


    prediction_score = np.mean(
        values[
            np.isfinite(values)
        ]
    )


    return (
        prediction_score,
        nrmse,
        successful_queries
    )


# ============================================================
# 7. ANALYZE ORIGINAL DATA
# ============================================================

lambda_real, r2_real = (
    calculate_lyapunov(
        data=data,
        tau=tau,
        m=m,
        theiler_window=theiler_window,
        max_k=max_k,
        fit_start=fit_start,
        fit_end=fit_end,
        dt=dt
    )
)


(
    real_prediction_score,
    real_errors,
    real_queries
) = nonlinear_prediction_error(
    data=data,
    tau=tau,
    m=m,
    theiler_window=theiler_window,
    k_neighbors=k_neighbors,
    horizons=prediction_horizons,
    n_prediction_points=n_prediction_points
)


print("\n======================================")
print("ORIGINAL DATA")
print("======================================")

print(
    f"Largest Lyapunov exponent: "
    f"{lambda_real:.8f}"
)

print(
    f"Lyapunov fit R^2: "
    f"{r2_real:.6f}"
)


print("\nPrediction NRMSE:")

for horizon in prediction_horizons:

    print(
        f"Horizon {horizon:3d}: "
        f"{real_errors[horizon]:.6f}"
    )


print(
    "\nOverall prediction score:",
    f"{real_prediction_score:.6f}"
)

print(
    "Successful prediction points:",
    real_queries
)


# ============================================================
# 8. PHASE-RANDOMIZED SURROGATE TEST
# ============================================================

surrogate_scores = []

surrogate_errors_by_horizon = {
    h: []
    for h in prediction_horizons
}


print("\n======================================")
print("PHASE-RANDOMIZED SURROGATES")
print("======================================")


for i in range(
    n_surrogates
):

    surrogate = (
        phase_randomized_surrogate(
            data,
            rng
        )
    )


    (
        score,
        errors,
        successful_queries
    ) = nonlinear_prediction_error(
        data=surrogate,
        tau=tau,
        m=m,
        theiler_window=theiler_window,
        k_neighbors=k_neighbors,
        horizons=prediction_horizons,
        n_prediction_points=n_prediction_points
    )


    surrogate_scores.append(
        score
    )


    for horizon in prediction_horizons:

        surrogate_errors_by_horizon[
            horizon
        ].append(
            errors[horizon]
        )


    print(
        f"Surrogate {i+1:3d}: "
        f"prediction score = "
        f"{score:.6f}"
    )


surrogate_scores = np.asarray(
    surrogate_scores
)


surrogate_scores = (
    surrogate_scores[
        np.isfinite(
            surrogate_scores
        )
    ]
)


# ============================================================
# 9. SURROGATE SUMMARY
# ============================================================

print("\n======================================")
print("PREDICTION COMPARISON")
print("======================================")


print(
    f"Original prediction score: "
    f"{real_prediction_score:.6f}"
)

print(
    f"Mean surrogate score: "
    f"{np.mean(surrogate_scores):.6f}"
)

print(
    f"Median surrogate score: "
    f"{np.median(surrogate_scores):.6f}"
)

print(
    f"Minimum surrogate score: "
    f"{np.min(surrogate_scores):.6f}"
)

print(
    f"Maximum surrogate score: "
    f"{np.max(surrogate_scores):.6f}"
)


print("\nMean surrogate NRMSE by horizon:")

for horizon in prediction_horizons:

    values = np.array(
        surrogate_errors_by_horizon[
            horizon
        ]
    )

    print(
        f"Horizon {horizon:3d}: "
        f"{np.nanmean(values):.6f}"
    )


# ============================================================
# 10. EMPIRICAL P-VALUE
#
# LOWER prediction error is the interesting direction.
#
# Ask:
#
# How many randomized surrogates predict as well as
# or better than the original?
# ============================================================

number_as_good_or_better = np.sum(
    surrogate_scores
    <= real_prediction_score
)


p_value = (
    number_as_good_or_better
    + 1
) / (
    len(surrogate_scores)
    + 1
)


print("\n======================================")
print("SIGNIFICANCE TEST")
print("======================================")


print(
    "Number of surrogates with prediction error "
    "≤ original:"
)

print(
    number_as_good_or_better
)


print(
    f"Empirical one-sided p-value: "
    f"{p_value:.4f}"
)


better_than_percent = (
    np.sum(
        surrogate_scores
        >
        real_prediction_score
    )
    /
    len(
        surrogate_scores
    )
) * 100


print(
    f"Original data predicts better than "
    f"{better_than_percent:.2f}% "
    f"of phase-randomized surrogates."
)


# ============================================================
# 11. INTERPRETATION
# ============================================================

print("\n======================================")
print("INTERPRETATION")
print("======================================")


print(
    f"Real lambda = "
    f"{lambda_real:.8f}"
)

print(
    f"Real nonlinear prediction score = "
    f"{real_prediction_score:.6f}"
)


if p_value <= 0.01:

    print(
        "\nThe real signal is substantially more "
        "short-term predictable in reconstructed state "
        "space than the phase-randomized surrogates."
    )

    print(
        "This is strong evidence for nonlinear temporal "
        "structure beyond the preserved linear "
        "power-spectrum/autocorrelation structure."
    )


elif p_value <= 0.05:

    print(
        "\nThe real signal is significantly more "
        "short-term predictable than the "
        "phase-randomized surrogates."
    )

    print(
        "This provides evidence for nonlinear "
        "temporal structure."
    )


else:

    print(
        "\nThe real signal is not substantially more "
        "predictable than the phase-randomized "
        "surrogates."
    )

    print(
        "This test therefore does not provide strong "
        "evidence against the linear stochastic null."
    )


print(
    "\nA positive Lyapunov exponent plus significantly "
    "better local prediction than phase-randomized "
    "surrogates is much more meaningful than comparing "
    "Lyapunov magnitudes directly."
)

Number of data points: 30000

ORIGINAL DATA
Largest Lyapunov exponent: 0.01557959
Lyapunov fit R^2: 0.997412

Prediction NRMSE:
Horizon   1: 0.015236
Horizon   5: 0.019082
Horizon  10: 0.023561
Horizon  20: 0.031968
Horizon  40: 0.086079

Overall prediction score: 0.035185
Successful prediction points: 2000

PHASE-RANDOMIZED SURROGATES
Surrogate   1: prediction score = 0.522165
Surrogate   2: prediction score = 0.530662
Surrogate   3: prediction score = 0.542914
Surrogate   4: prediction score = 0.538018
Surrogate   5: prediction score = 0.528688
Surrogate   6: prediction score = 0.537935
Surrogate   7: prediction score = 0.541282
Surrogate   8: prediction score = 0.526779
Surrogate   9: prediction score = 0.515829
Surrogate  10: prediction score = 0.531036
Surrogate  11: prediction score = 0.522295
Surrogate  12: prediction score = 0.519681
Surrogate  13: prediction score = 0.542540
Surrogate  14: prediction score = 0.531244
Surrogate  15: prediction score = 0.525055
Surrogate  16: pr